# Axon break detection — N3 (placeholder)

**N3 data is not available yet.** This notebook mirrors the N1 pipeline (no preprocessing before Frangi). Update the `directories` paths below once the N3 CZI files exist, then run. See `results/README.md`.

In [ ]:
from pylibCZIrw import czi as pyczi
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
from skimage.morphology import skeletonize, disk, dilation
from skimage.filters import frangi, gaussian, threshold_otsu
from scipy import ndimage
from scipy.spatial import cKDTree
from skimage.measure import label
from scipy.stats import kruskal
from itertools import combinations


In [ ]:
# ── I/O helpers (no extra preprocessing before Frangi) ────────────────────────

def _load(path, channel=0):
    """Max-Z projection over all Z planes at T=0, channel 0 (tubulin)."""
    with pyczi.open_czi(path) as cz:
        n_z = cz.total_bounding_box['Z'][1]
        planes = [np.squeeze(cz.read(plane={'C': channel, 'Z': z, 'T': 0}))
                  for z in range(n_z)]
    return np.stack(planes).max(axis=0)

def _norm(img):
    f = img.astype(float)
    return (f - f.min()) / (f.max() - f.min() + 1e-8)

def _n_zplanes(path):
    with pyczi.open_czi(path) as cz:
        return cz.total_bounding_box['Z'][1]


In [ ]:
# ── Skeleton helpers ───────────────────────────────────────────────────────────

def find_skeleton_endpoints(skeleton):
    kernel = np.ones((3, 3), dtype=np.uint8)
    kernel[1, 1] = 0
    nc = ndimage.convolve(skeleton.astype(np.uint8), kernel, mode='constant', cval=0)
    return np.argwhere((skeleton > 0) & (nc == 1))

def filter_small_components(skeleton, min_size=15):
    labeled = label(skeleton, connectivity=2)
    sizes   = np.bincount(labeled.ravel())
    keep    = np.where(sizes >= min_size)[0]
    keep    = keep[keep != 0]
    return np.isin(labeled, keep)


def preprocess_frangi(img, sigmas=(1, 2, 3, 4, 5), beta=0.5):
    """Frangi vesselness filter, no morphological closing (keeps real break gaps)."""
    img_smooth = gaussian(_norm(img), sigma=1)
    tube_map   = frangi(img_smooth, sigmas=sigmas, beta=beta, black_ridges=False)
    binary     = tube_map > threshold_otsu(tube_map)
    return binary, tube_map

def _make_skeleton(img, sigmas=(1, 2, 3, 4, 5)):
    binary, tube_map = preprocess_frangi(img, sigmas=sigmas)
    return skeletonize(binary), binary, tube_map


In [ ]:
# ── Skeleton metrics ───────────────────────────────────────────────────────────

def compute_metrics(skeleton, min_component_size=15, gap_radius=30):
    """
    skel_length      : total skeleton pixels
    n_components     : connected fragments after removing tiny spurs
    frag_rate        : n_components / skel_length x 1000
    continuity       : skel_length / n_components  (mean fragment length px)
    short_frac       : fraction of fragments shorter than 50 px
    endpoint_rate    : n_endpoints / skel_length x 1000
    gap_bridges      : number of components that merge when skeleton is dilated by gap_radius
    gap_bridge_rate  : gap_bridges / skel_length x 1000
    """
    skel_clean  = filter_small_components(skeleton, min_size=min_component_size).astype(bool)
    lbl         = label(skel_clean, connectivity=2)
    n_comp      = int(lbl.max())
    skel_length = int(skel_clean.sum())
    if n_comp == 0 or skel_length == 0:
        return dict(skel_length=0, n_components=0, frag_rate=0., continuity=0.,
                    short_frac=0., endpoint_rate=0., gap_bridges=0, gap_bridge_rate=0.)
    sizes           = np.bincount(lbl.ravel())[1:]
    frag_rate       = n_comp / skel_length * 1000
    continuity      = skel_length / n_comp
    short_frac      = float((sizes < 50).sum() / n_comp)
    endpoints       = find_skeleton_endpoints(skel_clean)
    endpoint_rate   = len(endpoints) / skel_length * 1000
    n_after         = int(label(dilation(skel_clean, disk(gap_radius)), connectivity=2).max())
    gap_bridges     = max(0, n_comp - n_after)
    gap_bridge_rate = gap_bridges / skel_length * 1000
    return dict(skel_length=skel_length, n_components=n_comp, frag_rate=frag_rate,
                continuity=continuity, short_frac=short_frac,
                endpoint_rate=endpoint_rate, gap_bridges=gap_bridges,
                gap_bridge_rate=gap_bridge_rate)


In [ ]:
# ── Break detection — direction + gap clarity filters ─────────────────────────

def _compute_tangent(skel_pts, ep, window=15):
    dists = np.linalg.norm(skel_pts - ep, axis=1)
    nearby = skel_pts[dists <= window]
    if len(nearby) < 3:
        return None
    centroid = nearby.mean(axis=0)
    vec = np.array(ep, dtype=float) - centroid
    n = np.linalg.norm(vec)
    return vec / n if n > 1e-8 else None

def _gap_is_clear(binary_mask, pt1, pt2, n_samples=30, border_frac=0.15, max_overlap=0.10):
    rs = np.linspace(pt1[0], pt2[0], n_samples)
    cs = np.linspace(pt1[1], pt2[1], n_samples)
    skip = max(1, int(n_samples * border_frac))
    rs = np.round(rs[skip:-skip]).astype(int)
    cs = np.round(cs[skip:-skip]).astype(int)
    if len(rs) == 0:
        return True
    rs = np.clip(rs, 0, binary_mask.shape[0] - 1)
    cs = np.clip(cs, 0, binary_mask.shape[1] - 1)
    return float(binary_mask[rs, cs].mean()) <= max_overlap

def find_broken_lines(skeleton, binary_mask=None, min_gap=10, max_gap=80,
                      min_component_size=15, angle_threshold=40,
                      max_gap_overlap=0.10, tangent_window=15):
    """
    Find axon break pairs using two filters:
      1. Direction: both endpoint tangents must be aligned (same axon)
      2. Gap clarity: the path between them must be dark (no crossing axon)
    """
    skeleton   = filter_small_components(skeleton, min_size=min_component_size).astype(bool)
    endpoints  = find_skeleton_endpoints(skeleton)
    if len(endpoints) < 2:
        return [], endpoints
    skel_pts   = np.argwhere(skeleton)
    comp_map   = label(skeleton, connectivity=2)
    ep_labels  = np.array([comp_map[r, c] for r, c in endpoints])
    cos_thresh = np.cos(np.radians(angle_threshold))
    tree       = cKDTree(endpoints)
    candidates = []
    for i, j in tree.query_pairs(r=max_gap):
        if ep_labels[i] == ep_labels[j]:
            continue
        dist = float(np.linalg.norm(endpoints[i] - endpoints[j]))
        if dist < min_gap:
            continue
        gap_vec = (endpoints[j].astype(float) - endpoints[i]) / dist
        ti = _compute_tangent(skel_pts, endpoints[i], window=tangent_window)
        tj = _compute_tangent(skel_pts, endpoints[j], window=tangent_window)
        if ti is not None and tj is not None:
            if np.dot(ti, gap_vec) < cos_thresh:   continue
            if np.dot(-tj, gap_vec) < cos_thresh:  continue
            if abs(np.dot(ti, tj)) < cos_thresh:   continue
        if binary_mask is not None:
            if not _gap_is_clear(binary_mask, tuple(endpoints[i]), tuple(endpoints[j]),
                                 max_overlap=max_gap_overlap):
                continue
        candidates.append((dist, i, j))
    candidates.sort()
    used = set(); breaks = []
    for dist, i, j in candidates:
        if i not in used and j not in used:
            used.update([i, j])
            breaks.append({'pt1': tuple(endpoints[i]),
                           'pt2': tuple(endpoints[j]),
                           'distance': dist})
    return breaks, endpoints


In [ ]:
# ── Per-image analysis, CSV table, comparison plot (no image display) ────────

def analyze_directory(directory, min_component_size=15, gap_radius=30, sigmas=(1, 2, 3, 4, 5),
                      min_gap=10, max_gap=80, angle_threshold=40,
                      max_gap_overlap=0.10, tangent_window=15):
    files = sorted(f for f in os.listdir(directory) if f.endswith('.czi'))
    rows = []
    for fname in files:
        fpath = os.path.join(directory, fname)
        img = _load(fpath)
        skel, binary, _ = _make_skeleton(img, sigmas)
        m = compute_metrics(skel, min_component_size, gap_radius)
        breaks, eps = find_broken_lines(
            skel, binary_mask=binary, min_gap=min_gap, max_gap=max_gap,
            min_component_size=min_component_size, angle_threshold=angle_threshold,
            max_gap_overlap=max_gap_overlap, tangent_window=tangent_window)
        row = {'filename': fname, 'n_zplanes': _n_zplanes(fpath),
               'n_breaks': len(breaks), 'n_endpoints': len(eps)}
        row.update(m)
        rows.append(row)
    return rows


def analyze_all(directories, **kwargs):
    all_rows = []
    for cond, directory in directories.items():
        for row in analyze_directory(directory, **kwargs):
            all_rows.append({'condition': cond, **row})
    return pd.DataFrame(all_rows)


def plot_comparison(df, save_path, gap_radius=30, min_component_size=15):
    """4-panel strip chart: frag_rate, endpoint_rate, gap_bridge_rate, continuity."""
    labels = list(dict.fromkeys(df['condition']))
    x_pos  = np.arange(len(labels))
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    panels = [
        ('frag_rate',       'Frags / 1k skel px',     'Frag rate  (more = more breaks)'),
        ('endpoint_rate',   'Endpoints / 1k skel px', 'Endpoint rate  (more = more breaks)'),
        ('gap_bridge_rate', f'Gap bridges / 1k px\n(dilation {gap_radius}px)',
                            'Gap-bridge rate  (more = more closeable breaks)'),
        ('continuity',      'Mean segment length (px)', 'Continuity  (more = healthier)'),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, (key, ylabel, title) in zip(axes, panels):
        groups = [df.loc[df['condition'] == l, key].to_numpy() for l in labels]
        for xi, (color, g) in enumerate(zip(colors, groups)):
            jitter = np.random.default_rng(0).uniform(-0.12, 0.12, size=len(g))
            ax.scatter(xi + jitter, g, s=70, color=color, zorder=3, alpha=0.85)
            mean = g.mean()
            sem  = g.std(ddof=1) / np.sqrt(len(g)) if len(g) > 1 else 0
            ax.plot([xi - .25, xi + .25], [mean, mean], color=color, lw=2.5, zorder=4)
            ax.errorbar(xi, mean, yerr=sem, fmt='none', color=color, capsize=5, lw=1.8, zorder=4)
        if all(len(g) >= 2 for g in groups):
            _, p_kw = kruskal(*groups)
            ax.text(0.98, 0.98, f'KW p={p_kw:.3f}', transform=ax.transAxes,
                    ha='right', va='top', fontsize=8, style='italic',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
            y_max = max(g.max() for g in groups)
            y_r   = y_max * 0.08; y_b = y_max * 1.05
            for i_, j_ in combinations(range(len(labels)), 2):
                if len(groups[i_]) >= 2 and len(groups[j_]) >= 2:
                    _, p2 = kruskal(groups[i_], groups[j_])
                    sig = '***' if p2 < 0.001 else '**' if p2 < 0.01 else '*' if p2 < 0.05 else 'ns'
                    xm  = (x_pos[i_] + x_pos[j_]) / 2
                    yl  = y_b + (abs(i_ - j_) - 1) * y_r
                    ax.plot([x_pos[i_], x_pos[i_], x_pos[j_], x_pos[j_]],
                            [yl, yl + y_r * .4, yl + y_r * .4, yl], lw=1.2, color='black')
                    ax.text(xm, yl + y_r * .5, sig, ha='center', va='bottom', fontsize=10)
        ax.set_xticks(x_pos); ax.set_xticklabels(labels, fontsize=12)
        ax.set_ylabel(ylabel, fontsize=10); ax.set_title(title, fontsize=11)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
    fig.suptitle(f"Axon integrity — min_size={min_component_size}, gap_radius={gap_radius}px",
                 fontsize=12)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved comparison plot to {save_path}")


## Directories and run

In [ ]:
directories = {
    # TODO: update these once N3 data is available (paths assumed to mirror N1's layout)
    "bmi": "/Users/ctuna/Desktop/axon_codes/N3/x63 bmi tubulin iii_data/",
    "Ni":  "/Users/ctuna/Desktop/axon_codes/N3/x63 Ni tubulin_data/",
    "scr": "/Users/ctuna/Desktop/axon_codes/N3/x63 scr tubIII_data/",
}

RESULTS_DIR = '/Users/ctuna/Desktop/axon_codes/results/N3'
CSV_PATH  = os.path.join(RESULTS_DIR, "csv", "axon_break_metrics_N3.csv")
PLOT_PATH = os.path.join(RESULTS_DIR, "figures", "axon_break_comparison_N3.png")

df = analyze_all(directories, gap_radius=30)
df.to_csv(CSV_PATH, index=False)
print(f"Saved metrics for {len(df)} images to {CSV_PATH}")

plot_comparison(df, PLOT_PATH, gap_radius=30)
df
